# Le tour des grands nombres — quelle écriture courte pour quelle croissance ?

**Série Complexity — accrétion 07b** · Public : **Recherche** · Hommage fondateur : **Tibor Rado** (1962)

**Navigation** : [<< Complexity-07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) | [Index de la série](README.md) | [Complexity-07c — Le bilan mathocalypse →](Complexity-07c-LeBilanMathocalypse-Python.ipynb)

Cette accrétion prolonge le [07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) : la question du 07 était *quelles séquences peut-on écrire court* ; la nôtre est son miroir exact — **quels nombres peut-on écrire court, et pourquoi les écrire court n'aide pas à les calculer**. Du googol au nombre de Rayo, une même échelle de mesure : la hiérarchie à croissance rapide. Le pont avec la série Compression (EPIC #18706) est le fil rouge : un googolisme est une écriture préfixe de quelques symboles pour une valeur hors d'atteinte — la face miroir de l'inégalité de Kraft.

Conventions de la série : chaque fait est **Mesuré** (sortie de ce carnet) ou **Cité** (source nominative). Sources principales : S. Aaronson, « The Mathocalypse » (7/10/2026, mise à jour 12/10/2026), archivée au gisement Symbolic/ ; Rado (1962), Lin-Radó (1965), Brady (1983), Marxen-Buntrock (1990), bbchallenge (2024) pour le castor affairé.

## 1. Googol, googolplex — quand l'échelle des puissances devient muette

Kasner (1940) : $10^{100}$ — le nom vient du neveu de neuf ans de Kasner, Milton Sirotta, et le googolplex fut défini dans le même ouvrage (*Mathematics and the Imagination*, Kasner & Newman) comme un 1 suivi de zéros « jusqu'à ce que l'écrivain se fatigue » : la fatigue comme seule borne, déjà, avant toute notation d'itération.


In [1]:
import math

googol = 10 ** 100
print(f"googol = 10**100 : {len(str(googol))} chiffres (Mesure)")
print(f"googolplex = 10**googol : {googol} chiffres exactement (Mesure - l'exposant EST le compte de chiffres)")
print(f"Ecrire le googolplex en decimal : {googol/400:.2e} pages de 400 chiffres (Mesure)")

googol = 10**100 : 101 chiffres (Mesure)
googolplex = 10**googol : 10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000 chiffres exactement (Mesure - l'exposant EST le compte de chiffres)
Ecrire le googolplex en decimal : 2.50e+97 pages de 400 chiffres (Mesure)


### Lecture du résultat — l'écriture décimale dépasse l'univers avant le nombre

La sortie montre le googol encore tenable (101 chiffres, mesuré), mais le googolplex déjà hors d'atteinte : son exposant **est** son compte de chiffres, et l'écrire en décimal demanderait 2,50e+97 pages de 400 chiffres — la mesure dit l'impossibilité. Le nombre est parfaitement défini ; son écriture ne tient dans aucun support physique. Toute la suite du carnet dérive de cette asymétrie : chercher **l'écriture courte d'une croissance**, jamais le nombre écrit.


## 2. Ackermann (1928) — la récursion qui sort des primitives

Ackermann prouve que toute fonction construite par composition et récursion primitives reste en deçà de certaines fonctions récursives totales : la sienne, où l'on descend sur deux arguments, échappe au schéma primitif — la réponse à Hilbert que la récursion générale dépasse strictement la récursion primitive.


In [2]:
import sys
sys.setrecursionlimit(200000)

def ackermann(m, n):
    """Fonction d'Ackermann-Péter : A(0,n)=n+1 ; A(m,0)=A(m-1,1) ; A(m,n)=A(m-1,A(m,n-1))."""
    if m == 0:
        return n + 1
    if n == 0:
        return ackermann(m - 1, 1)
    return ackermann(m - 1, ackermann(m, n - 1))

print("Table mesurée A(m,n) :")
for m in range(4):
    print(f"  m={m}: {[ackermann(m, n) for n in range(5)]}")
print(f"A(3,5) = {ackermann(3,5)} ; identité 2**(n+3)-3 vérifiée : {2**(5+3)-3 == ackermann(3,5)} (Mesure)")
print("A(4,1) = 65533 (Cité - 2↑↑3-3 ; la récursion naïve y met ~6 minutes : l'explosion EST la leçon)")

Table mesurée A(m,n) :
  m=0: [1, 2, 3, 4, 5]
  m=1: [2, 3, 4, 5, 6]
  m=2: [3, 5, 7, 9, 11]
  m=3: [5, 13, 29, 61, 125]
A(3,5) = 253 ; identité 2**(n+3)-3 vérifiée : True (Mesure)
A(4,1) = 65533 (Cité - 2↑↑3-3 ; la récursion naïve y met ~6 minutes : l'explosion EST la leçon)


### Lecture du résultat — la sortie du monde primitive-récursif

Les trois premières lignes de la table restent polynomiales ($m=0,1,2$), puis $m=3$ saute à $2^{n+3}-3$ : A(3,5) = 253 et l'identité vérifiée True. Mais A(4,1) = 65 533 est **cité**, non mesuré — la récursion naïve y met ~6 minutes, et l'explosion est la leçon : chaque ligne itère la précédente, et la diagonale croît plus vite que toute fonction construite par composition et récursion primitives (Ackermann 1928). La table s'arrête à $m=3$ non par choix mais par contrainte.


## 3. Flèches de Knuth (1976) — chaque flèche itère la précédente

$a \uparrow^k b$ : la flèche $k$ itère la flèche $k-1$. $3\uparrow\uparrow 3 = 3^{3^3}$ tient encore à l'écran ; $3\uparrow\uparrow\uparrow 3$ dépasse déjà toute écriture décimale.

In [3]:
import math

def knuth(a, k, b):
    """a ↑^k b, récursif ; petits arguments seulement."""
    if k == 1:
        return a ** b
    if b == 0:
        return 1
    return knuth(a, k - 1, knuth(a, k, b - 1))

v = knuth(3, 2, 3)
print(f"3↑↑3 = 3^3^3 = {v} (Mesure, {len(str(v))} chiffres)")
# 3↑↑4 = 3^(3↑↑3) : la LONGUEUR se calcule par log10 sans evaluer le nombre
long_3up4 = v * math.log10(3)
print(f"3↑↑4 : {math.ceil(long_3up4):,} chiffres (Mesure par log10 - le nombre n'est pas évalué)")
print(f"3↑↑↑3 = 3↑↑(3↑↑3) : tour de hauteur {v} (Mesure de la hauteur)")

3↑↑3 = 3^3^3 = 7625597484987 (Mesure, 13 chiffres)
3↑↑4 : 3,638,334,640,025 chiffres (Mesure par log10 - le nombre n'est pas évalué)
3↑↑↑3 = 3↑↑(3↑↑3) : tour de hauteur 7625597484987 (Mesure de la hauteur)


### Lecture du résultat — chaque flèche saute un étage d'écriture

3↑↑3 = 7 625 597 484 987 tient encore en 13 chiffres (mesuré), mais 3↑↑4 compte déjà 3 638 334 640 025 chiffres — l'organe ne l'évalue plus, il mesure son $\log_{10}$ ; et 3↑↑↑3 n'est plus un nombre affiché mais une **tour de hauteur 7 625 597 484 987**, la hauteur étant elle-même la valeur mesurée de la ligne précédente. Le geste se répète à chaque flèche : ce que $\uparrow^k$ écrivait en une expression, $\uparrow^{k+1}$ ne l'écrit plus qu'en descripteur.


## 4. Chaînes fléchées de Conway — une échelle au-dessus

Conway (1996) : $a 	o b = a^b$, puis la chaîne allongée réécrit les flèches. La notation vit dans *The Book of Numbers* (Conway & Guy) et sert à indexer des quantités combinatoires que les flèches seules n'expriment plus — c'est l'outil qui encadrera le nombre de Graham ci-dessous.


In [4]:
def conway3(a, b, c):
    """Chaînes à 3 éléments : a→b→c = a ↑^c b (Conway-Guy 1996).
    Garde : ne jamais évaluer les géants - la chaîne 3→3→3 vaut 3↑↑↑3,
    une tour de hauteur 3↑↑3 : on rend la HAUTEUR, pas la valeur."""
    return knuth(a, c, b)

print(f"2→2→2 = {conway3(2,2,2)} (Mesure ; identité 2↑↑2 = 4)")
print(f"3→3→3 = 3↑↑↑3 : tour de hauteur {knuth(3,2,3)} (Mesure de la hauteur, jamais évaluée)")
print("4→4→4→4 (Cité, Conway-Guy 1996) : dépasse toute expression de Knuth de longueur fixe — l'écriture EST le contenu")

2→2→2 = 4 (Mesure ; identité 2↑↑2 = 4)
3→3→3 = 3↑↑↑3 : tour de hauteur 7625597484987 (Mesure de la hauteur, jamais évaluée)
4→4→4→4 (Cité, Conway-Guy 1996) : dépasse toute expression de Knuth de longueur fixe — l'écriture EST le contenu


### Lecture du résultat — la chaîne mange les flèches

2→2→2 = 4 retombe sur l'identité 2↑↑2 (mesure de cohérence), puis 3→3→3 = 3↑↑↑3 : la chaîne de Conway à trois éléments n'est qu'un renommage des flèches de Knuth, la hauteur mesurée est identique à la cellule précédente. Le saut vient du quatrième élément : 4→4→4→4 **dépasse toute expression de Knuth de longueur fixe** (cité, Conway-Guy 1996). La sortie ne montre pas un nombre : elle montre un changement de primitive — l'écriture elle-même est devenue le contenu.


## 5. Le nombre de Graham (1971) — quand le nombre de flèches devient le message

Graham et Rothschild (1971) majorent un nombre de Ramsey sur les hypergraphes : la borne $G = g_{64}$, avec $g_1 = 3\uparrow\uparrow\uparrow\uparrow 3$ puis $g_{n+1} = 3\uparrow^{g_n} 3$. La cascade des **descripteurs** — la hauteur de chaque tour — est le seul récit tenable : la valeur ne s'écrit pas.

In [5]:
# Descripteurs uniquement - la valeur de g_1 n'est meme pas representable
h3upup3 = knuth(3, 2, 3)            # 3↑↑3 = 7625597484987 (Mesure)
print(f"g_1 = 3↑↑↑↑3 = 3↑↑↑(3↑↑↑3)")
print(f"  or 3↑↑↑3 = tour de hauteur 3↑↑3 = tour de hauteur {h3upup3:,} (Mesure de la hauteur)")
print(f"  donc g_1 = tour de 3 de hauteur (tour de 3 de hauteur (tour de hauteur {h3upup3:,})) (Descripteur mesuré)")
print("g_2 = 3↑^g_1 3 : le NOMBRE DE FLÈCHES vaut déjà g_1 - la notation elle-même dépasse l'univers (Cité)")
print("G = g_64 (Cité, Graham-Rothschild 1971) ; la vraie valeur du nombre de Ramsey reste encadrée :")
print("  borne inférieure connue de l'ordre de 13, borne supérieure G - l'écart est abyssal (Cité)")

g_1 = 3↑↑↑↑3 = 3↑↑↑(3↑↑↑3)
  or 3↑↑↑3 = tour de hauteur 3↑↑3 = tour de hauteur 7,625,597,484,987 (Mesure de la hauteur)
  donc g_1 = tour de 3 de hauteur (tour de 3 de hauteur (tour de hauteur 7,625,597,484,987)) (Descripteur mesuré)
g_2 = 3↑^g_1 3 : le NOMBRE DE FLÈCHES vaut déjà g_1 - la notation elle-même dépasse l'univers (Cité)
G = g_64 (Cité, Graham-Rothschild 1971) ; la vraie valeur du nombre de Ramsey reste encadrée :
  borne inférieure connue de l'ordre de 13, borne supérieure G - l'écart est abyssal (Cité)


### Lecture du résultat — le nombre de flèches devient le message

Le descripteur mesuré de $g_1$ s'imbrique déjà sur trois niveaux (tour de hauteur tour de hauteur 7 625 597 484 987), mais $g_2$ retourne la situation : son **nombre de flèches** vaut $g_1$ — la notation ne sert plus à écrire le nombre, elle l'indexe. $G = g_{64}$ (cité, Graham-Rothschild 1971), et l'ironie finale de la sortie : la valeur du nombre de Ramsey qui motive $G$ reste encadrée entre ~13 et $G$ — le plus grand nombre jamais utilisé dans une preuve publiée coexiste avec une borne inférieure minuscule.


## 6. TREE(3) — le jeu de Kruskal : écrire des arbres sans se répéter

Le jeu de Friedman sur le théorème de Kruskal (1960) : une séquence d'arbres enracinés étiquetés $T_1, T_2, \ldots$ où $T_i$ a au plus $i$ sommets, étiquettes parmi $\{1, \ldots, n\}$, et **aucun $T_i$ ne se plonge dans un $T_j$ ultérieur**. Le théorème de Kruskal garantit que le jeu finit toujours ; la question est *quand*. Les valeurs canoniques : TREE(1) = 1, TREE(2) = 3, et TREE(3) dépasse le nombre de Graham (Cité — Friedman, posts FOM ; le plongement exact de sa définition est plus fin que celui de notre jouet ci-dessous).

**Notre jouet** implémente la variante suivante, déclarée : arbres non ordonnés ; le plongement exige racine $\to$ racine, étiquettes **exactes**, ancêtres préservés, et deux enfants distincts envoyés dans deux sous-arbres d'enfants **distincts**. Le jouet mesure l'explosion (le phénomène wqo de Kruskal), pas les valeurs canoniques de Friedman.

In [6]:
from functools import lru_cache

def canonique(t):
    """Clé canonique d'un arbre (etiquette, enfants) : enfants triés - arbre non ordonné."""
    return (t[0], tuple(sorted(canonique(e) for e in t[1])))

@lru_cache(maxsize=None)
def arbres(k, n):
    """Tous les arbres enracinés non ordonnés de taille EXACTEMENT k, étiquettes 1..n."""
    if k == 1:
        return tuple((e, ()) for e in range(1, n + 1))
    vus, out = set(), []
    for e in range(1, n + 1):
        for combo in multisets(k - 1, n):
            t = (e, combo)
            c = canonique(t)
            if c not in vus:
                vus.add(c)
                out.append(t)
    return tuple(out)

def multisets(reste, n):
    """Multi-ensembles d'arbres (taille totale 'reste'), enfants triés par clé canonique."""
    if reste == 0:
        return [()]
    out = []
    def rec(r, taille_max, prefix):
        if r == 0:
            out.append(tuple(prefix))
            return
        for k in range(1, min(r, taille_max) + 1):
            for t in arbres(k, n):
                if not prefix or canonique(t) >= canonique(prefix[-1]):
                    rec(r - k, k, prefix + [t])
    rec(reste, reste, [])
    return out

def plonge_racine(s, t):
    """s se plonge racine->racine dans t (étiquettes exactes, frères distincts
    -> sous-arbres d'enfants distincts). C'est la relation du JEU déclarée ci-dessus."""
    if s[0] != t[0]:
        return False
    enf_s, enf_t = s[1], t[1]
    if len(enf_s) > len(enf_t):
        return False
    def contient(u, a):
        """L'arbre a se plonge racine->racine dans UN sommet du sous-arbre u."""
        if plonge_racine(a, u):
            return True
        return any(contient(v, a) for v in u[1])
    def essayer(i, restants):
        if i == len(enf_s):
            return True
        for j in restants:
            if contient(enf_t[j], enf_s[i]):
                if essayer(i + 1, [r for r in restants if r != j]):
                    return True
        return False
    return essayer(0, list(range(len(enf_t))))

def jeu_valide(seq):
    """True ssi aucune paire i<j ne viole la règle (T_i ne plonge pas dans T_j)."""
    return all(not plonge_racine(seq[i], seq[j]) for i in range(len(seq)) for j in range(i + 1, len(seq)))

def plus_longue(n, taille_max, budget=100000):
    """Longueur maximale (exhaustif borné) du jeu à n étiquettes, |T_i| <= i."""
    pools = {k: arbres(k, n) for k in range(1, taille_max + 1)}
    compteur = [0]
    best = [0]
    def dfs(seq):
        compteur[0] += 1
        if compteur[0] >= budget:
            raise RuntimeError("budget")
        best[0] = max(best[0], len(seq))
        i = len(seq) + 1
        for k in range(1, min(i, taille_max) + 1):
            for t in pools[k]:
                if all(not plonge_racine(s, t) for s in seq):
                    dfs(seq + [t])
    try:
        dfs([])
        return best[0], compteur[0], True
    except RuntimeError:
        return best[0], compteur[0], False

for n, tmax in ((1, 3), (2, 5)):
    lg, noeuds, termine = plus_longue(n, tmax)
    print(f"Jeu jouet à {n} étiquette(s) : plus longue séquence = {lg} (exhaustif={termine}, {noeuds} noeuds) (Mesure)")
lg, noeuds, termine = plus_longue(3, 4, budget=4000)
print(f"Jeu jouet à 3 étiquettes : recherche interrompue à {noeuds} noeuds, déjà {lg} coups")
print(f"  (Mesure de l'explosion - l'énumération sature avant de conclure ; TREE(3) canonique >> Graham, Cité)")

Jeu jouet à 1 étiquette(s) : plus longue séquence = 1 (exhaustif=True, 2 noeuds) (Mesure)


Jeu jouet à 2 étiquette(s) : plus longue séquence = 9 (exhaustif=True, 373 noeuds) (Mesure)


Jeu jouet à 3 étiquettes : recherche interrompue à 4000 noeuds, déjà 18 coups
  (Mesure de l'explosion - l'énumération sature avant de conclure ; TREE(3) canonique >> Graham, Cité)


### Lecture du résultat — l'énumération sature avant le nombre

Le jeu jouet à 1 étiquette conclut vite (séquence 1, 2 nœuds), à 2 étiquettes l'exhaustif conclut séquence 9 en 373 nœuds — mais à 3 étiquettes la recherche **s'interrompt à 4 000 nœuds, déjà 18 coups** : la mesure est celle de l'explosion, pas de la conclusion. C'est la différence de nature avec les sections précédentes : pour Graham on ne peut plus écrire le nombre, pour TREE(3) on ne peut même pas terminer l'**énumération du jeu** qui le définit (TREE(3) ≫ Graham, cité).


## 7. Le castor affairé (Rado, 1962) — le nombre que ZFC ne peut pas atteindre

Rado demande : parmi les machines de Turing à $n$ états et 2 symboles qui s'arrêtent, combien de 1 au maximum sur la bande ($\Sigma(n)$), et en combien de pas ($S(n)$) ? La fonction n'est pas calculable — mais les petites valeurs se **simulent**. Subtilité mesurable dès $n=3$ : la machine qui maximise $\Sigma$ n'est pas celle qui maximise $S$ — deux records, deux machines (Lin-Radó 1965). On exécute les championnes de $\Sigma$ (tables Cité : Rado 1962, Lin-Radó 1965, Brady 1983 ; page « Busy beaver », Wikipedia, consultée le 14/10/2026).

In [7]:
def simule(transitions, max_steps=1000):
    """Simule une machine 2-symboles. transitions[(etat, symbole_lu)] =
    (symbole_ecrit, 'L'|'R', prochain_etat) ; prochain_etat None = halt.
    Retourne (nb de 1, nb de pas) ou None si la garde de pas est atteinte."""
    bande = {0: 0}
    pos, etat, pas = 0, 'A', 0
    while etat is not None:
        if pas >= max_steps:
            return None
        s = int(bande.get(pos, 0))
        ecrit, direction, suivant = transitions[(etat, s)]
        bande[pos] = int(ecrit)
        pos += 1 if direction == 'R' else -1
        etat = suivant
        pas += 1
    return sum(1 for v in bande.values() if v == 1), pas

CHAMPIONNES = {
    2: {('A',0):('1','R','B'), ('A',1):('1','L','B'),
        ('B',0):('1','L','A'), ('B',1):('1','R',None)},
    3: {('A',0):('1','R','B'), ('A',1):('1','R',None),
        ('B',0):('0','R','C'), ('B',1):('1','R','B'),
        ('C',0):('1','L','C'), ('C',1):('1','L','A')},
    4: {('A',0):('1','R','B'), ('A',1):('1','L','B'),
        ('B',0):('1','L','A'), ('B',1):('0','L','C'),
        ('C',0):('1','R',None),('C',1):('1','L','D'),
        ('D',0):('1','R','D'), ('D',1):('0','R','A')},
}
ETABLIS = {2: (4, 6), 3: (6, 14), 4: (13, 107)}

for n in (2, 3, 4):
    uns, pas = simule(CHAMPIONNES[n])
    ok = (uns, pas) == ETABLIS[n]
    print(f"n={n} : championne de Σ simulée -> Σ={uns}, S={pas} ; valeurs établies {ETABLIS[n]} ; coïncidence : {ok} (Mesure)")
print("n=3 : le record de PAS S(3)=21 est établi par une AUTRE machine de Lin-Radó,")
print("  qui n'écrit que 5 uns - championne de Σ et record S vivent sur deux machines (Cité)")
print("n=5 : Σ=4098, S=47 176 870 (Cité - Marxen-Buntrock 1990 ; preuve complétée et formalisée en Rocq, bbchallenge 2024)")
print("n=6 : Σ(6) > 2↑↑↑5 (Cité - bbchallenge, survey Michel 2026)")
print("BB(10) > nombre de Graham ; BB(335) indépendant de ZFC (Cité - Ketchersid 2026, via agents Claude Code,")
print("  battant BB(13)>Graham et BB(393)/ZFC de Wade ; la lignée part d'Aaronson-Yedidia BB(8000) il y a une décennie)")

n=2 : championne de Σ simulée -> Σ=4, S=6 ; valeurs établies (4, 6) ; coïncidence : True (Mesure)
n=3 : championne de Σ simulée -> Σ=6, S=14 ; valeurs établies (6, 14) ; coïncidence : True (Mesure)
n=4 : championne de Σ simulée -> Σ=13, S=107 ; valeurs établies (13, 107) ; coïncidence : True (Mesure)
n=3 : le record de PAS S(3)=21 est établi par une AUTRE machine de Lin-Radó,
  qui n'écrit que 5 uns - championne de Σ et record S vivent sur deux machines (Cité)
n=5 : Σ=4098, S=47 176 870 (Cité - Marxen-Buntrock 1990 ; preuve complétée et formalisée en Rocq, bbchallenge 2024)
n=6 : Σ(6) > 2↑↑↑5 (Cité - bbchallenge, survey Michel 2026)
BB(10) > nombre de Graham ; BB(335) indépendant de ZFC (Cité - Ketchersid 2026, via agents Claude Code,
  battant BB(13)>Graham et BB(393)/ZFC de Wade ; la lignée part d'Aaronson-Yedidia BB(8000) il y a une décennie)


### Lecture du résultat — les championnes simulées coïncident, puis décollent

Pour $n = 2, 3, 4$, la simulation retrouve exactement les valeurs établies (Σ = 4, 6, 13 ; S = 6, 14, 107 ; coïncidence True trois fois) : le carnet vérifie son outil sur le terrain connu. La nuance arrive à $n = 3$ : le record de **pas** S(3) = 21 appartient à une autre machine que la championne de Σ — les deux records vivent sur deux machines distinctes (Lin-Radó, cité). Puis l'énumération ne suit plus : $n = 5$ vient d'une preuve (Marxen-Buntrock 1990, formalisée en Rocq, bbchallenge 2024) et $n = 6$ franchit 2↑↑↑5 — comme l'annonçait la section 7, la fonction Σ croît plus vite que toute théorie calculable : ces nombres ne se mesurent plus, ils se démontrent.


## 8. Le nombre de Rayo (2007) — la limite logique du duel

Le « grand nombre duel » du MIT (2007) : Rayo nomme *le plus petit nombre plus grand que tout nombre définissable en moins de $10^{100}$ symboles de logique du premier ordre sur l'univers des ensembles*. Ce nombre dépasse toute fonction nommable — casteur affairé compris — précisément parce que sa définition quantifie **sur les définitions**. C'est le point où le jeu du googolisme se ferme sur lui-même : écrire court ce qu'aucune théorie ne peut calculer.

(Cité — A. Rayo, janvier 2007 ; expositions standard du duel, notamment le blog de A. Aaronson.)

## Gap Mathlib

À la date de rédaction (14/10/2026), aucune formalisation de la fonction d'Ackermann, des flèches de Knuth ou du castor affairé n'est référencée dans les sources de ce dépôt (`git grep ackermann/uparrow/busyBeaver` : vide ; Mathlib vendored non disponible localement ce cycle — vérification non exhaustive, à reprendre). Le pont formel le plus proche : la preuve bbchallenge 2024 que $S(5) = 47\,176\,870$ a été **formalisée en Rocq**, et côté indécidabilité la lignée BB(8000) → BB(393) → BB(335)/ZFC (Yedidia-Aaronson, Wade, Ketchersid) rejoint les questions de Gödel suivies par l'EPIC #15066.

## Exercice 1 — la longueur de la tour, sans la tour

Écrire `longueur_2upup(k)` : le **nombre de chiffres** de $2\uparrow\uparrow k$, sans évaluer $2\uparrow\uparrow 6$ (qui aurait un nombre inimaginable de chiffres). Indice : $nb(2^m) = \lfloor m \cdot \log_{10} 2 \rfloor + 1$, et $2\uparrow\uparrow k = 2^{(2\uparrow\uparrow (k-1))}$ ; pour $k \leq 5$ l'entier $2\uparrow\uparrow 5 = 2^{65536}$ tient en machine, au-delà il faut faire la multiplication en précision étendue (`decimal`).

In [8]:
from decimal import Decimal, getcontext

def longueur_2upup(k):
    # TODO etudiant : nombre de chiffres de 2↑↑k, exact, sans evaluer 2↑↑6
    # Etape 1 : k <= 5 -> len(str(2↑↑k)) directement (2↑↑5 = 2**65536 tient en machine)
    # Etape 2 : k = 6 -> floor(2↑↑5 * log10(2)) + 1, en Decimal a precision suffisante
    #           (le RESULTAT a ~19729 chiffres : prevoir getcontext().prec en consequence)
    # Etape 3 : verifier contre le contrôle : len(str(2**65536)) doit sortir pour k=5
    return None

# Controle attendu : longueur_2upup(4) == 5 (2↑↑4 = 65536) et longueur_2upup(5) == 19729

## Exercice 2 — une séquence du jouet, à la main

Le carnet a mesuré (exhaustivement) que le jeu jouet à 2 étiquettes admet des séquences de longueur jusqu'à 9. Construire **à la main** une séquence valide `SEQ` de longueur 4 ou plus (une liste d'arbres sous forme de tuples `(etiquette, enfants)`) et la vérifier avec `jeu_valide(SEQ)` fourni en §6. Rappel : $T_i$ a au plus $i$ sommets.

In [9]:
def reponse_exo2():
    # TODO etudiant : construire SEQ = [...] (liste d'arbres (etiquette, tuple-enfants))
    # valide pour le jeu a 2 etiquettes, longueur >= 4, puis return jeu_valide(SEQ)
    # Indice : commencer par la feuille (1, ()) et ne jamais permettre a un arbre
    # anterieur de se plonger racine->racine dans le suivant (cf. plonge_racine)
    return None

## Exercice 3 — retrouver le casteur à 2 états par énumération

Il n'existe que $12^4 = 20\,736$ machines à 2 états (chaque case (état, symbole) $	o$ 2 symboles $\times$ 2 directions $\times$ 3 suites A/B/halt). Énumérer et retrouver $\Sigma(2) = 4$ et $S(2) = 6$.

In [10]:
import itertools

def castor_2etats():
    # TODO etudiant : enumerer les 20 736 machines (produit cartesien des 12 choix
    # pour chacune des 4 cases (A,0),(A,1),(B,0),(B,1)), simuler chacune avec
    # simule(t, max_steps=50) fourni en §7, et retourner (max des Sigma, max des S)
    # sur les machines qui s'arretent.
    # Etape 1 : suites = ['AL','AR','BL','BR','A0L', ...] ou toute encodage propre
    #           (12 suites possibles : ecrit 0/1, direction L/R, etat A/B/halt)
    # Etape 2 : itertools.product(suites, repeat=4) -> table de transitions
    # Etape 3 : max sur les resultats non None de simule()
    return None

# Controle attendu : castor_2etats() == (4, 6)

## Conclusion — écrire court, ne pas calculer

La lignée de ce carnet tient en une phrase : **chaque étage écrit plus court ce qu'il rend plus incalculable**. Le googol s'écrit $10^{100}$ ; le googolplex, une exponentielle de plus ; Ackermann, une récursion ; Knuth et Conway, quelques flèches ; Graham, un nombre de flèches qui est lui-même un géant ; TREE(3) et le casteur, des objets combinatoires dont la théorie (Kruskal, Rado) garantit l'existence tout en interdisant le calcul ; Rayo, enfin, quantifie sur les définitions elles-mêmes.

C'est le miroir exact du [07 — Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) : là-bas, on demandait quelles **séquences** acceptent une description courte ; ici, des nombres entiers jouent le même rôle — et l'inégalité de Kraft (série Compression, #18706) dit la même chose d'un code : préfixes courts et non-ambigus, valeurs hors d'atteinte. Jusqu'au mur final : $BB(335)$ échappe à ZFC — écrire court touche la limite de la théorie elle-même (EPIC #15066).